# Signyales — Gesture Recognition Model Training

This notebook trains the TinyML gesture recognition model from data collected with
`pi-side/tools/collect_gesture.py`, and exports `model.tflite`, `labels.json`,
and `scaler.json` directly into the Pi hub's `model/` directory.

**Before running this notebook:**
1. Record at least a few gestures with `collect_gesture.py` (20+ repetitions per gesture is a
   reasonable starting point — see the WARNING this notebook prints if you have less).
2. Place recordings under `data/<label>/rep_NNNN.npz`, one subfolder per gesture.

The three exported files are a **contract** with `signyales_hub/model_runtime.py` on the Pi —
changing the feature order, window size, or label ordering here without also updating the hub
(or vice versa) will not raise an error, it will just make the model silently wrong.

In [ ]:
import sys
sys.path.insert(0, "src")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

from dataset import load_dataset
from model import build_model
from export import export_artifacts

print("TensorFlow version:", tf.__version__)

## 1. Load collected data

Each `.npz` file is one repetition of one gesture: a `(window_size, 22)` array already
resampled to a fixed length by `collect_gesture.py`. The 22 features, in order, are:

`left_flex(5), right_flex(5), left_accel(3), left_gyro(3), right_accel(3), right_gyro(3)`

In [ ]:
DATA_DIR = "data"

X, y, labels = load_dataset(DATA_DIR)
print(f"Loaded {X.shape[0]} samples, shape {X.shape}, across {len(labels)} labels:")
print(labels)

samples_per_class = np.bincount(y)
for label, count in zip(labels, samples_per_class):
    print(f"  {label}: {count} samples")

if X.shape[0] < 20 * len(labels):
    print("\nWARNING: fewer than ~20 samples per class on average. Collect more data "
          "before trusting these results — a small dataset can look deceptively good "
          "on its own test split while not generalizing to a different session.")

## 2. Visualize a few raw examples

Sanity-check the data before spending time training on it — this is the fastest way to
notice a labeling mistake, a sensor that wasn't connected, or a recording that's mostly noise.

In [ ]:
fig, axes = plt.subplots(1, min(3, len(labels)), figsize=(15, 4), sharey=True)
if len(labels) == 1:
    axes = [axes]

for ax, label_idx in zip(axes, range(min(3, len(labels)))):
    sample_idx = np.where(y == label_idx)[0][0]
    ax.plot(X[sample_idx])
    ax.set_title(labels[label_idx])
    ax.set_xlabel("frame")

axes[0].set_ylabel("feature value")
plt.suptitle("One example window per gesture (all 22 features overlaid)")
plt.tight_layout()
plt.show()

## 3. Train / validation / test split

Stratified so every class is represented proportionally in each split. The scaler
(mean/std) is computed **only** on the training split — fitting it on the full dataset
would leak test-set statistics into training and make the reported accuracy optimistic.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

flattened_train = X_train.reshape(-1, X_train.shape[-1])
mean = flattened_train.mean(axis=0)
std = flattened_train.std(axis=0)
std[std == 0] = 1.0  # guard against a constant feature column

def apply_scaler(X):
    return (X - mean) / std

X_train_n = apply_scaler(X_train)
X_val_n = apply_scaler(X_val)
X_test_n = apply_scaler(X_test)

## 4. Build and train the model

A compact 1D-CNN — enough capacity to separate a handful of gestures without overfitting
a hand-collected dataset, and small enough to quantize cleanly for edge deployment later.

In [ ]:
model = build_model(window_size=X.shape[1], feature_count=X.shape[2], num_classes=len(labels))
model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=10, restore_best_weights=True
)

history = model.fit(
    X_train_n, y_train,
    validation_data=(X_val_n, y_val),
    epochs=60,
    batch_size=16,
    callbacks=[early_stop],
    verbose=2,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["accuracy"], label="train")
ax1.plot(history.history["val_accuracy"], label="val")
ax1.set_title("Accuracy")
ax1.set_xlabel("epoch")
ax1.legend()

ax2.plot(history.history["loss"], label="train")
ax2.plot(history.history["val_loss"], label="val")
ax2.set_title("Loss")
ax2.set_xlabel("epoch")
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Evaluate on the held-out test set

This is the Recognition Accuracy Matrix referenced in the methodology — the confusion
matrix below shows exactly which gestures the model confuses with which others, which is
more useful for deciding what to fix than the overall accuracy number alone.

In [ ]:
test_loss, test_acc = model.evaluate(X_test_n, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  (loss: {test_loss:.4f})")

y_pred = model.predict(X_test_n, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Recognition Accuracy Matrix")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=labels, zero_division=0))

## 6. Export for deployment

Writes `model.tflite`, `labels.json`, and `scaler.json` directly into the Pi hub's `model/`
directory — the hub will pick these up on its next run with no further changes needed.

Set `QUANTIZE = True` to export an int8-quantized model instead of float32. Quantization
mainly matters if this model ever needs to run on the ESP32-class hardware described as a
future direction in the proposal; for inference on the Raspberry Pi 5 itself, float32 is
simpler to debug and the Pi has enough headroom that the speed difference is unlikely to
matter — profile with `pi-side/tools/benchmark_inference.py` before assuming
otherwise.

In [ ]:
QUANTIZE = False
OUTPUT_DIR = "../pi-side/model"

export_artifacts(
    model, labels, mean, std, OUTPUT_DIR,
    quantize=QUANTIZE,
    representative_data=X_train_n if QUANTIZE else None,
)

## 7. Sanity-check the exported model

Loads the just-exported `model.tflite` back in (independently of the Keras model still in
memory) and confirms it produces the same prediction on a real test sample. This catches
export bugs — a wrong quantization setting, a mismatched input shape — before they show up
as a confusing failure on the Pi three steps removed from this notebook.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=f"{OUTPUT_DIR}/model.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test_n[0]
x = np.expand_dims(sample, axis=0).astype(input_details[0]["dtype"])
interpreter.set_tensor(input_details[0]["index"], x)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]["index"])[0]

keras_output = model.predict(np.expand_dims(sample, axis=0), verbose=0)[0]

print("Keras prediction:  ", labels[keras_output.argmax()], keras_output)
print("TFLite prediction: ", labels[tflite_output.argmax()], tflite_output)
assert labels[keras_output.argmax()] == labels[tflite_output.argmax()], \
    "Keras and exported TFLite models disagree — check the export step above."
print("\nOK: exported model matches the trained model's predictions.")